# Membina Aplikasi Penjanaan Imej (Azure OpenAI)

Ada lebih daripada hanya penjanaan teks dengan LLM. Anda juga boleh menjana imej daripada penerangan teks. Imej sebagai mod adalah berguna dalam MedTech, seni bina, pelancongan, pembangunan permainan, pemasaran, dan banyak lagi. Dalam pelajaran ini kita bekerja dengan model **GPT Image** hari ini di Microsoft Foundry.

## Matlamat pembelajaran

Pada akhir pelajaran ini anda akan dapat:

- Menerangkan apa itu penjanaan imej dan di mana ia berguna.
- Memahami keluarga model `gpt-image` dan bagaimana ia berbeza daripada model lama DALL·E.
- Membina aplikasi penjanaan imej dan mengedit imej.

## Apa itu penjanaan imej?

Model penjanaan imej mencipta imej daripada permintaan teks. Model moden seperti `gpt-image` mempelajari hubungan antara teks dan imej semasa latihan, kemudian secara berperingkat menukar bunyi rawak menjadi imej yang sepadan dengan permintaan anda.

Dua keluarga model imej yang terkenal adalah:

- **`gpt-image` (OpenAI)** — generasi semasa yang digunakan dalam pelajaran ini. Ia menyokong penjanaan teks-ke-imej dan penyuntingan imej (inpainting dengan topeng).
- **Midjourney** — model pihak ketiga yang popular dengan perkhidmatan dan aliran kerja berasaskan Discord sendiri.

> Model imej OpenAI yang lebih lama — **DALL·E 2** dan **DALL·E 3** — adalah warisan. DALL·E 3 tidak lagi tersedia untuk pengedaran baru, dan ciri seperti `create_variation` hanya wujud dalam DALL·E 2. Gunakan model `gpt-image` untuk aplikasi baharu.

Di Microsoft Foundry, **`gpt-image-2`** adalah model imej terkini dan paling berupaya serta disyorkan secara lalai. `gpt-image-1.5` dan `gpt-image-1-mini` juga tersedia secara umum.

> **Penting:** model `gpt-image` memulangkan imej yang dijana sebagai **base64** (`b64_json`), bukan sebagai URL. Kod anda perlu menyahkod rentetan base64 kepada bait dan menyimpannya — tiada URL imej untuk dimuat turun.


## Membina aplikasi penjana imej pertama anda

Jadi apa yang diperlukan untuk membina aplikasi penjana imej? Anda memerlukan perpustakaan berikut:

- **python-dotenv**, anda sangat digalakkan menggunakan perpustakaan ini untuk menyimpan rahsia anda dalam fail *.env* jauh dari kod.
- **openai**, perpustakaan ini adalah yang akan anda gunakan untuk berinteraksi dengan API OpenAI.
- **pillow**, untuk bekerja dengan imej dalam Python.

Jika belum dilakukan, ikuti arahan di halaman [Microsoft Learn](https://learn.microsoft.com/azure/ai-foundry/openai/how-to/create-resource?pivots=web-portal&WT.mc_id=academic-105485-koreyst) untuk membuat sumber dan model Microsoft Foundry. Pilih **gpt-image-2** sebagai model (model imej Azure OpenAI terkini; DALL·E adalah warisan).

1. Buat fail *.env* dengan kandungan berikut:

    ```text
    AZURE_OPENAI_ENDPOINT=<your endpoint>
    AZURE_OPENAI_API_KEY=<your key>
    AZURE_OPENAI_DEPLOYMENT="gpt-image-2"
    ```

    Cari maklumat ini dalam portal Microsoft Foundry untuk sumber anda dalam bahagian "Deployments".



1. Kumpulkan perpustakaan di atas dalam fail yang dipanggil *requirements.txt* seperti berikut:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. Seterusnya, cipta persekitaran maya dan pasang perpustakaan tersebut:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Untuk Windows, gunakan arahan berikut untuk membuat dan mengaktifkan persekitaran maya anda:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. Tambahkan kod berikut dalam fail yang dipanggil *app.py*:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError

    # import dotenv
    dotenv.load_dotenv()

    # konfigurasi klien Azure OpenAI (Microsoft Foundry).
    # Model imej memerlukan versi API terkini — semak dokumentasi Foundry untuk versi yang model anda perlukan.
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )

    try:
        # Cipta imej menggunakan API penjanaan imej
        generation_response = client.images.generate(
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Masukkan teks arahan anda di sini
            size='1024x1024',
            n=1,
            model=os.environ['AZURE_OPENAI_DEPLOYMENT']   # contohnya "gpt-image-2"
        )
        # Tetapkan direktori untuk imej yang disimpan
        image_dir = os.path.join(os.curdir, 'images')

        # Jika direktori tidak wujud, buatnya
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # Mulakan jalan imej (nota jenis fail harus png)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # model gpt-image mengembalikan imej sebagai base64 (b64_json), bukan URL
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # Paparkan imej dalam penampil imej lalai
        image = Image.open(image_path)
        image.show()

    # tangkap pengecualian
    except BadRequestError as err:
        print(err)

    ```

Mari kita terangkan kod ini:

- Pertama, kami mengimport perpustakaan yang kami perlukan, termasuk perpustakaan OpenAI, perpustakaan dotenv, modul base64, dan perpustakaan Pillow.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    from openai import AzureOpenAI, BadRequestError
    ```

- Seterusnya, kami memuatkan pembolehubah persekitaran dari fail *.env*.

    ```python
    # import dotenv
    dotenv.load_dotenv()
    ```

- Selepas itu, kami mengkonfigurasi klien Azure OpenAI (Microsoft Foundry).

    ```python
    client = AzureOpenAI(
      azure_endpoint = os.environ["AZURE_OPENAI_ENDPOINT"],
      api_key=os.environ['AZURE_OPENAI_API_KEY'],
      api_version = "2025-04-01-preview"
      )
    ```

- Seterusnya, kami menjana imej:

    ```python
    generation_response = client.images.generate(
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # Masukkan teks arahan anda di sini
        size='1024x1024',
        n=1,
        model=os.environ['AZURE_OPENAI_DEPLOYMENT']
    )
    ```

    model `gpt-image` mengembalikan imej sebagai rentetan **base64** dalam `data[0].b64_json`. Kami nyahkodkan ia kepada bait dan menulisnya ke fail — tiada URL untuk dimuat turun.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- Akhir sekali, kami membuka imej dan menggunakan penonton imej standard untuk memaparkannya:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### Perincian lanjut mengenai penjanaan imej

Mari kita lihat parameter `images.generate`:

- **prompt**, ialah teks arahan yang digunakan untuk menjana imej. Di sini ia adalah "Arnab di atas kuda, memegang lollipop, di padang berkabus tempat tumbuh bunga daffodil".
- **size**, ialah saiz imej yang dijana (contohnya `1024x1024`, `1536x1024`, `1024x1536`, atau `"auto"`).
- **n**, ialah bilangan imej yang dijana. Di sini kami jana satu.
- **model**, ialah nama pengerahan model imej anda (contohnya `gpt-image-2`).

> Model imej tidak mengambil parameter `temperature` — itu kawalan penjanaan teks. Untuk mendapatkan variasi, panggil API sekali lagi; untuk mengurangkan variasi, buat arahan anda lebih spesifik.

## Keupayaan tambahan penjanaan imej

Anda telah melihat bagaimana menjana imej dengan beberapa baris Python. Model `gpt-image` juga boleh **mengedit** imej yang sedia ada. Dengan menyediakan imej, **topeng** pilihan (yang menandakan kawasan untuk diubah), dan arahan, anda boleh mengubah sebahagian imej — contohnya, tambah topi pada arnab kami.

```python
response = client.images.edit(
  model=os.environ['AZURE_OPENAI_DEPLOYMENT'],
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# suntingan juga dipulangkan sebagai base64
edited_image = base64.b64decode(response.data[0].b64_json)
```

Imej asas hanya mengandungi arnab; imej akhir mempunyai topi pada arnab.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Penafian**:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI [Co-op Translator](https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi kesilapan atau ketidaktepatan. Dokumen asal dalam bahasa asalnya harus dianggap sebagai sumber yang sahih. Untuk maklumat penting, terjemahan oleh manusia profesional adalah disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsir yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
